<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_1_autoencoders.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_1_autoencoders.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F04_representation%2Fnotebooks%2Fsesion_04_1_autoencoders.ipynb)

# SI7011 · Sesión 04 · 1. Autoencoders: ¿qué guarda un cuello de botella?
**Juan David Martínez Vargas · Universidad EAFIT**

Acompaña la **Parte 2** de las diapositivas. Aquí ninguna red ve una etiqueta mientras se entrena: la única señal es reconstruir la entrada. Las etiquetas aparecen solo al final, para **medir** qué tan útil quedó la representación $\mathbf h = g(\mathbf x)$.

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | autoencoder con $k = 2$ | el espacio latente se organiza sin etiquetas |
| 2 | $k = 2, 8, 32, 128$ | error de reconstrucción y *linear probe* sobre $\mathbf h$ |
| 3 | autoencoder **lineal** frente a PCA | el mismo subespacio |
| 4 | reconstrucciones lado a lado | la figura de la diapositiva |
| 5 | sin cuello de botella | la trampa de la identidad |

**Datos:** Fashion-MNIST, 70 000 imágenes de ropa de 28×28 en gris, 10 clases. Corre en CPU en unos 5 minutos (Colab, Kaggle, Lightning o local).

In [ ]:
import json, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision import datasets

torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__)

## Datos: Fashion-MNIST como vectores
Cada imagen se aplana en $\mathbf x \in \mathbb R^{784}$ con valores en $[0, 1]$. Las etiquetas se cargan, pero **el autoencoder nunca las usa**: solo sirven para la *linear probe* y para colorear gráficas.

In [ ]:
train_set = datasets.FashionMNIST('data', train=True, download=True)
test_set = datasets.FashionMNIST('data', train=False, download=True)
CLASSES = train_set.classes

X_train = train_set.data.reshape(-1, 784).float() / 255      # x ∈ [0, 1]^784
X_test = test_set.data.reshape(-1, 784).float() / 255
y_train, y_test = train_set.targets, test_set.targets
X_train.shape, X_test.shape

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(11, 1.6))
for c, ax in enumerate(axes):
    ax.imshow(X_train[(y_train == c).nonzero()[0, 0]].reshape(28, 28), cmap='gray_r')
    ax.set_title(CLASSES[c], fontsize=7); ax.axis('off')
plt.show()

## Herramientas: el autoencoder, el loop y la *linear probe*
- `autoencoder(k)` devuelve el encoder $g$: 784 → 256 → $k$ y el decoder $d$: $k$ → 256 → 784 (diapositiva *Encoder and decoder*).
- `train_ae` minimiza $L_i = \lVert \mathbf x_i - d(g(\mathbf x_i)) \rVert^2$ (promediado por píxel) con Adam, un número fijo de **pasos**, no de épocas, para que todos los modelos reciban el mismo trabajo.
- `probe(H_train, H_test)` congela $\mathbf h$ y entrena **una sola capa lineal** con las etiquetas (diapositiva *Measuring a representation*). Su accuracy de prueba mide cuánta información de la clase quedó en $\mathbf h$, y qué tan fácil es de leer.

In [ ]:
def autoencoder(k, hidden=256):
    g = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, k))      # encoder g: x ↦ h ∈ ℝ^k
    d = nn.Sequential(nn.Linear(k, hidden), nn.ReLU(), nn.Linear(hidden, 784))      # decoder d: h ↦ x̂
    return g, d

def train_ae(g, d, steps=4800, batch=128, lr=1e-3, snapshots=(), X_snap=None):
    # la semilla se fija antes de crear el modelo: misma inicialización y mismos lotes que la animación A01
    opt = torch.optim.Adam([*g.parameters(), *d.parameters()], lr=lr)
    fotos, step, t0 = {}, 0, time.time()
    if 0 in snapshots:
        with torch.no_grad(): fotos[0] = g(X_snap).numpy()
    while step < steps:
        for b in torch.randperm(len(X_train)).split(batch):
            xb = X_train[b]
            loss = ((d(g(xb)) - xb) ** 2).mean()          # J_𝓑 = promedio de ‖x − x̂‖² por píxel; sin etiquetas
            opt.zero_grad(); loss.backward(); opt.step()
            step += 1
            if step in snapshots:
                with torch.no_grad(): fotos[step] = g(X_snap).numpy()
            if step == steps: break
    print(f'{steps} pasos en {time.time() - t0:.0f} s')
    return fotos

@torch.no_grad()
def recon_mse(g, d, X=None):
    X = X_test if X is None else X
    return ((d(g(X)) - X) ** 2).mean().item()            # MSE de prueba, por píxel

def probe(H_train, H_test, steps=500):
    # linear probe: h congelado, una capa lineal, lote completo; se estandariza h con estadísticas de entrenamiento
    mu, sd = H_train.mean(0), H_train.std(0) + 1e-6
    Htr, Hte = (H_train - mu) / sd, (H_test - mu) / sd
    torch.manual_seed(0)
    head = nn.Linear(Htr.shape[1], 10)                  # ŷ = softmax(W h + b): lo único que ve etiquetas
    opt = torch.optim.Adam(head.parameters(), lr=1e-2)
    for _ in range(steps):
        loss = nn.functional.cross_entropy(head(Htr), y_train)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (head(Hte).argmax(1) == y_test).float().mean().item()

print(f'probe sobre los 784 píxeles: {probe(X_train, X_test):.3f}   (azar: 0.100)')

La *probe* sobre los píxeles crudos es la referencia: una capa lineal sobre $\mathbf x$ ya separa bastante bien Fashion-MNIST. Una representación de **pocas** dimensiones que se acerque a ese número guardó lo importante en mucho menos espacio.

## 1. Autoencoder con $k = 2$
**Predice antes de correr:** con solo dos números por imagen, ¿qué forma tendrá la nube de 1 000 imágenes de prueba en el plano $(h_1, h_2)$ al principio y al final del entrenamiento?

Guardamos $\mathbf h$ en 9 momentos del entrenamiento. Es el mismo modelo, la misma semilla y los mismos pasos de la animación de la diapositiva *The latent space organizes itself*.

In [ ]:
PASOS_FOTO = [0, 25, 75, 150, 300, 600, 1200, 2400, 4800]
idx_vis = torch.randperm(len(X_test), generator=torch.Generator().manual_seed(0))[:1000]

torch.manual_seed(0)
g2, d2 = autoencoder(2)
fotos = train_ae(g2, d2, snapshots=PASOS_FOTO, X_snap=X_test[idx_vis])

fig, axes = plt.subplots(1, 9, figsize=(16, 2.1))
for ax, s in zip(axes, PASOS_FOTO):
    ax.scatter(*fotos[s].T, s=2, c='0.4'); ax.set_title(f'paso {s}', fontsize=8); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('h = g(x) en el plano, sin color: el autoencoder no conoce las clases', fontsize=9, y=1.05); plt.show()

In [ ]:
# solo ahora coloreamos por clase
plt.figure(figsize=(5.5, 4.5))
sc = plt.scatter(*fotos[4800].T, s=4, c=y_test[idx_vis], cmap='tab10')
plt.legend(sc.legend_elements()[0], CLASSES, fontsize=6, loc='best', markerscale=0.7)
plt.title('las mismas 1 000 imágenes, coloreadas por clase'); plt.xlabel('$h_1$'); plt.ylabel('$h_2$'); plt.show()

Pantalones, bolsos y calzado forman grupos propios; camisas, abrigos y pullovers se mezclan, igual que se confunden a simple vista. Nada de eso se le dijo a la red: aparece porque, para reconstruir con dos números, conviene poner cerca las imágenes que se parecen.

## 2. ¿Cuánto debe medir el cuello de botella?
**Cambia solo $k$.** Para cada tamaño medimos dos cosas en prueba: el error de reconstrucción y la *linear probe* sobre $\mathbf h$.

**Predice:** ¿las dos columnas mejoran juntas al subir $k$?

In [ ]:
resultados, modelos = {}, {2: (g2, d2)}
for k in [2, 8, 32, 128]:
    if k not in modelos:
        torch.manual_seed(0)
        modelos[k] = autoencoder(k)
        train_ae(*modelos[k])
    g, d = modelos[k]
    with torch.no_grad():
        H_tr, H_te = g(X_train), g(X_test)
    resultados[f'AE k={k}'] = {'k': k, 'mse': recon_mse(g, d), 'probe': probe(H_tr, H_te)}
    print(f"k = {k:3d} · MSE {resultados[f'AE k={k}']['mse']:.4f} · probe {resultados[f'AE k={k}']['probe']:.3f}")

In [ ]:
ks = [2, 8, 32, 128]
fig, ax1 = plt.subplots(figsize=(5.5, 3.2))
ax1.plot(ks, [resultados[f'AE k={k}']['mse'] for k in ks], 'o-', color='tab:blue'); ax1.set_ylabel('MSE de prueba', color='tab:blue')
ax2 = ax1.twinx(); ax2.plot(ks, [resultados[f'AE k={k}']['probe'] for k in ks], 's-', color='tab:orange')
ax2.set_ylabel('probe (accuracy)', color='tab:orange'); ax2.grid(False)
ax1.set_xscale('log', base=2); ax1.set_xticks(ks, ks); ax1.set_xlabel('k (tamaño de h)'); plt.show()

Las dos mejoran al subir $k$, pero no al mismo ritmo. El error de reconstrucción sigue bajando con fuerza hasta $k = 128$, mientras que cada duplicación de $k$ le suma a la *probe* cada vez menos, y con $k = 128$ ya queda cerca de la *probe* sobre los 784 píxeles. Las dimensiones extra se gastan sobre todo en redibujar detalles (bordes, texturas, el brillo exacto) que no cambian la clase. La sección 3 lo muestra aún más claro con el mismo $k$.

## 3. Un autoencoder lineal aprende lo mismo que PCA
**Cambia las capas:** quitamos la ReLU y la capa oculta. Queda $g(\mathbf x) = W_g \mathbf x + \mathbf b_g$ y $d(\mathbf h) = W_d \mathbf h + \mathbf b_d$, con $k = 32$. La diapositiva *A linear autoencoder is PCA* dice que, con MSE, aprende el mismo subespacio que las 32 primeras componentes principales.

In [ ]:
K = 32
# PCA: las k direcciones de mayor varianza, con la media de entrenamiento
mu = X_train.mean(0)
U, S, Vt = torch.linalg.svd(X_train - mu, full_matrices=False)
V = Vt[:K].T                                              # 784 × k, columnas ortonormales
pca_rec = lambda X: (X - mu) @ V @ V.T + mu
mse_pca = ((pca_rec(X_test) - X_test) ** 2).mean().item()

torch.manual_seed(0)
g_lin, d_lin = nn.Linear(784, K), nn.Linear(K, 784)       # sin φ: g y d lineales
train_ae(g_lin, d_lin, steps=9600, lr=3e-3)
mse_lin = recon_mse(g_lin, d_lin)

# ¿el mismo subespacio? cosenos de los ángulos principales entre las columnas de W_d y las de V
Q, _ = torch.linalg.qr(d_lin.weight.detach())             # base ortonormal del espacio que genera el decoder
cosenos = torch.linalg.svdvals(V.T @ Q)
print(f'MSE de prueba · PCA {mse_pca:.4f} · AE lineal {mse_lin:.4f} · AE con ReLU {resultados["AE k=32"]["mse"]:.4f}')
print(f'cosenos de los ángulos principales: {(cosenos > 0.9).sum()} de {K} por encima de 0.9, mínimo {cosenos.min():.3f}  (1 = misma dirección)')
var = S ** 2
print(f'varianza de la componente 32 frente a la 33: difieren en {100 * (var[K - 1] - var[K]) / var[K - 1]:.1f} %')

El autoencoder lineal llega al error de PCA y casi todas sus direcciones coinciden con las de PCA (cosenos cercanos a 1). La que falta no es un error: las componentes 32 y 33 tienen casi la misma varianza, así que cualquier mezcla de las dos reconstruye igual de bien. Sin $\varphi$, un autoencoder es PCA aprendido por descenso de gradiente. Lo único nuevo de un autoencoder profundo es la no linealidad, y por eso con ReLU el error baja más.

Para la figura de la diapositiva, también medimos la *probe* sobre las 32 coordenadas de PCA.

In [ ]:
resultados['PCA k=32'] = {'k': K, 'mse': mse_pca, 'probe': probe((X_train - mu) @ V, (X_test - mu) @ V)}
resultados['AE lineal k=32'] = {'k': K, 'mse': mse_lin,
                                 'probe': probe(g_lin(X_train).detach(), g_lin(X_test).detach())}
a, b = resultados['PCA k=32'], resultados['AE k=32']
print(f"k = 32 · PCA: MSE {a['mse']:.4f}, probe {a['probe']:.3f} · AE con ReLU: MSE {b['mse']:.4f}, probe {b['probe']:.3f}")
print(f"el AE reconstruye con {100 * (1 - b['mse'] / a['mse']):.0f} % menos error, y la probe cambia {100 * (b['probe'] - a['probe']):+.1f} puntos")

Con el mismo $k$, el autoencoder no lineal reconstruye mucho mejor que PCA, y aun así la *probe* casi no cambia. Es la diapositiva *Reconstruction is not the goal*: **menor error de reconstrucción no implica una representación más útil para la tarea.**

## 4. Reconstrucciones lado a lado
Las mismas 8 imágenes de prueba (la primera de 8 clases distintas) pasadas por cada modelo. Es la figura de la diapositiva. Los números y las reconstrucciones se guardan en `resultados_ae.json` y `reconstrucciones_ae.npz`.

In [ ]:
idx8 = torch.stack([(y_test == c).nonzero()[0, 0] for c in [0, 1, 2, 3, 5, 7, 8, 9]])
filas = {'original': X_test[idx8]}
with torch.no_grad():
    for k in [2, 8, 32]:
        g, d = modelos[k]; filas[f'AE k={k}'] = d(g(X_test[idx8]))
    filas['PCA k=32'] = pca_rec(X_test[idx8])

fig, axes = plt.subplots(len(filas), 8, figsize=(9, 5.8))
for r, (nombre, imgs) in enumerate(filas.items()):
    for c in range(8):
        axes[r, c].imshow(imgs[c].reshape(28, 28).clamp(0, 1), cmap='gray_r', vmin=0, vmax=1); axes[r, c].axis('off')
    extra = '' if nombre == 'original' else f"\nMSE {resultados[nombre]['mse']:.3f} · probe {resultados[nombre]['probe']:.2f}"
    axes[r, 0].text(-4, 14, nombre + extra, ha='right', va='center', fontsize=8)
plt.show()

json.dump({'indices': idx8.tolist(), 'resultados': resultados}, open('resultados_ae.json', 'w'), indent=1)
np.savez_compressed('reconstrucciones_ae.npz', **{n.replace(' ', '_'): v.numpy() for n, v in filas.items()})

Con $k = 2$ todas las prendas se vuelven una silueta promedio de su grupo; con $k = 32$ aparecen las formas, pero no las texturas finas. El autoencoder con ReLU y PCA con el mismo $k$ se ven parecidos, aunque el primero tiene menor error.

## 5. La trampa de la identidad
**Quitamos el cuello de botella:** $g$ y $d$ lineales de 784 a 784. ¿Qué error de reconstrucción alcanza? ¿Y qué *probe*?

In [ ]:
torch.manual_seed(0)
g_id, d_id = nn.Linear(784, 784), nn.Linear(784, 784)
train_ae(g_id, d_id, steps=2400)
with torch.no_grad():
    H_tr, H_te = g_id(X_train), g_id(X_test)
p_id = probe(H_tr, H_te)
print(f'k = 784 · MSE {recon_mse(g_id, d_id):.5f} · probe {p_id:.3f}   (probe sobre los píxeles: {probe(X_train, X_test):.3f})')

El error se acerca a cero, pero la *probe* sobre $\mathbf h$ queda donde estaba la de los píxeles: $\mathbf h$ es solo $\mathbf x$ en otras coordenadas. Una pérdida casi nula **no** prueba que se aprendió algo. Para que reconstruir enseñe, hay que impedir la copia: un cuello de botella (aquí) o una entrada corrompida (notebook 2).

## Tu turno · 10 minutos
1. Entrena $k = 8$ con `hidden=64` y con `hidden=1024`. ¿Qué cambia más, el MSE o la *probe*?
2. Repite la sección 1 con otra semilla. ¿Es la misma nube? ¿Los mismos grupos? ¿Qué te dice eso sobre interpretar cada eje $h_1, h_2$?
3. En la sección 2, cambia la *probe* lineal por $k$ vecinos más cercanos sobre $\mathbf h$ (notebook 5). ¿El orden entre los valores de $k$ se mantiene?

**Para pensar.** Si el objetivo final es clasificar ropa, ¿por qué no entrenar directamente con las etiquetas? ¿En qué situación preferirías el autoencoder? (pista: diapositiva *The question for today*).

In [ ]:
# Experimento 1: hidden=64 y hidden=1024 con k = 8

### Referencias
- Hinton y Salakhutdinov (2006), *Reducing the Dimensionality of Data with Neural Networks*, Science.
- Baldi y Hornik (1989), *Neural Networks and Principal Component Analysis: Learning from Examples Without Local Minima*, Neural Networks (autoencoder lineal y PCA).
- Xiao, Rasul y Vollgraf (2017), *Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms*, arXiv:1708.07747.